# Fase 2 — Análisis por preguntas de investigación**Proyecto:** Rentabilidad de modelos de alquiler en San Vicente del RaspeigCada sección responde una pregunta del documento `preguntas_analisis.md`. Al final de cada una hay una celda para que escribas la conclusión con tus palabras: eso es lo que acaba en el informe y en el dashboard.## Las cuatro estrategias comparadas1. **Residencial anual** — piso completo, un inquilino, todo el año2. **Estudiantil por habitaciones** — 3 habitaciones sueltas; curso académico (sep-jun) y verano por separado3. **Turístico** — Airbnb/Booking, precio por noche × noches ocupadas4. **Mixto** — curso académico por habitaciones + verano turísticoTodas sobre el **mismo arquetipo** y el mismo precio de compra, para que la comparación sea justa.---

## 0. Carga y parámetros

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snssns.set_theme(style="white", palette="bright")pd.set_option("display.max_columns", 30)df = pd.read_csv("../limpio/dataset_unificado.csv", sep=";")arq = df[df["es_comparable_arquetipo"] == True]# --- Datos observados (rellena con lo que calculaste en la fase 1) ---PRECIO_COMPRA = None          # TODO: mediana de venta del arquetipoALQUILER_MES = None           # TODO: mediana de alquiler residencial del arquetipoPRECIO_HABITACION = None      # TODO: mediana del precio por habitaciónPRECIO_NOCHE = None           # TODO: mediana del precio/noche turístico comparable# --- Supuestos (NO son datos: hipótesis del modelo) ---COMUNIDAD_MES = 70.0IBI_PCT_CATASTRAL = 0.00767RATIO_CATASTRAL_MERCADO = 0.55ITP_MAS_GASTOS_COMPRA = 0.11SEGURO_IMPAGO_PCT = 0.065OCUPACION_RESIDENCIAL = 0.95OCUPACION_ESTUDIANTIL_CURSO = 0.95OCUPACION_ESTUDIANTIL_VERANO = 0.30MESES_CURSO = 10   # sep-junMESES_VERANO = 2   # jul-ago

### Gastos comunes**Pregunta:** ¿qué gastos paga el propietario en cualquier estrategia?IBI (tipo oficial del municipio 0,767% sobre valor catastral, que estimamos al 55% del de mercado) y comunidad de propietarios.

In [ ]:
# TODO: calcula los gastos fijos anuales y la inversión total (compra + gastos de adquisición)# Comprobación: gastos fijos ≈ 1.844 EUR/año, inversión total ≈ 264.124 EUR

---## P1.1 — Entre residencial y estudiantil, ¿cuál rinde más?Esta es la comparación **más sólida del proyecto**, porque no depende del supuesto de ocupación turística.Ojo con una trampa: no asumas que el estudiantil cobra 12 meses. En verano la demanda universitaria desaparece — ese fue justamente un error de una versión anterior del modelo que invertía el resultado.

In [ ]:
# TODO: calcula para RESIDENCIAL: ingreso bruto anual, gastos, ingreso neto, ROI# Recuerda aplicar la ocupación residencial y el seguro de impago

In [ ]:
# TODO: calcula lo mismo para ESTUDIANTIL, separando curso y verano# bruto = precio_hab * 3 hab * meses_curso * ocupacion_curso#       + precio_hab * 3 hab * meses_verano * ocupacion_verano

In [ ]:
# TODO: monta una tabla comparativa y un gráfico de barras# Comprobación: residencial ≈ 3,30% ROI y estudiantil ≈ 2,44% (sobre inversión total)

**Tu conclusión (P1.1):** ¿cuál gana y por qué? Explica el mecanismo, no solo el número._(tu respuesta)_

---## P1.2 — ¿Cuántas noches necesita el turístico para compensar?Aquí está la clave metodológica del proyecto. En vez de asumir una ocupación y calcular el ROI (que haría depender la conclusión de una cifra que nos inventamos), **invertimos la pregunta**: partimos de los precios observados y despejamos las noches necesarias.Además calcularás un **caso suelo** que usa solo el coste documentado (la comisión de plataforma) y pone a cero limpieza, suministros, gestión y mantenimiento. Ese suelo es un límite inferior real: no depende de ninguna estimación.

In [ ]:
# Estructuras de costes a probar (todas son SUPUESTOS salvo la comisión)COSTES = {    "suelo (solo comisión)": dict(comision=0.12, gestion=0.0, mantenimiento=0.0,                                   limpieza_estancia=0.0, noches_estancia=4, suministros_mes=0.0),    "optimista": dict(comision=0.05, gestion=0.0, mantenimiento=0.04,                      limpieza_estancia=45, noches_estancia=5, suministros_mes=100),    "base": dict(comision=0.12, gestion=0.10, mantenimiento=0.05,                 limpieza_estancia=50, noches_estancia=4, suministros_mes=120),    "pesimista": dict(comision=0.15, gestion=0.18, mantenimiento=0.07,                      limpieza_estancia=60, noches_estancia=3, suministros_mes=150),}# TODO: escribe una función neto_turistico(noches_mes, costes) que devuelva el# ingreso neto mensual. Ten en cuenta que la limpieza escala con las ESTANCIAS,# no con las noches: n_estancias = noches / noches_estancia

In [ ]:
# TODO: despeja cuántas noches/mes hacen falta para igualar el neto del residencial# Como neto(n) es lineal en n, puedes resolverlo analíticamente o con un barrido fino# Comprobación: el caso suelo debería darte ≈ 6,9 noches/mes

In [ ]:
# TODO: grafica el neto mensual frente a las noches (una línea por estructura de costes)# y añade líneas horizontales con el neto del residencial y del estudiantil

**Tu conclusión (P1.2):** ¿cuántas noches hacen falta y qué significa el caso suelo? ¿Te parece alcanzable ese número en San Vicente?_(tu respuesta)_

---## P2 — ¿Cómo cambia el ranking según el escenario?Ahora sí, calcula el ROI de las cuatro estrategias en los tres escenarios (cada escenario mueve a la vez la ocupación turística y los costes).Ocupaciones asumidas: pesimista 45%, base 60%, optimista 77%. **Solo el 77% tiene origen documentado** (proxy de Alicante ciudad); los otros dos los fijamos como rango plausible.

In [ ]:
# TODO: calcula el ROI de las 4 estrategias en los 3 escenarios# Para el mixto: curso por habitaciones + verano turístico

In [ ]:
# TODO: gráfico de barras agrupadas (estrategia en el eje, escenario como leyenda)

**Tu conclusión (P2):** ¿el orden se mantiene entre escenarios? ¿Qué variables explican el cambio?_(tu respuesta)_

---## P3 — ¿En cuántos años se amortiza la compra?Payback simple: años hasta que el ingreso neto acumulado cubre la inversión total.Cuando escribas la conclusión, menciona qué **no** incluye este cálculo (revalorización, inflación, valor temporal del dinero, coste de oportunidad). Es lo que separa "payback del alquiler" de "rentabilidad de la inversión".

In [ ]:
# TODO: calcula el payback de cada estrategia en cada escenario

**Tu conclusión (P3):** ¿qué estrategia amortiza antes? ¿Qué limitaciones tiene esta métrica?_(tu respuesta)_

---## P4 — ¿Cómo es el mercado turístico de San Vicente?Este hallazgo salió al intentar ampliar la muestra. Analiza la composición del parque turístico del municipio.

In [ ]:
# TODO: sobre las filas de mercado == "turistico", analiza:#   - cuántas propiedades únicas hay en San Vicente#   - cuántas son villa/chalet frente a piso/apartamento (columna marca_calidad)#   - cuántas son realmente comparables al arquetipo# Cruza el resultado con el registro oficial: filas de mercado == "vut_registro"

**Tu conclusión (P4):** ¿qué implica esta composición para la estrategia turística? ¿Por qué n=4 no es un fallo de método?_(tu respuesta)_

---## P5 — ¿Hay estacionalidad en los precios turísticos?Se midió consultando las mismas propiedades en tres fechas (febrero, octubre, julio). La clave metodológica: comparar **la misma vivienda** consigo misma, no viviendas distintas entre sí.

In [ ]:
# TODO: carga limpio/turistico_estacionalidad.csv y analiza la variación jul vs feb# ¿Se comportan igual las villas que los pisos?

**Tu conclusión (P5):** ¿qué tipo de demanda turística sugiere el patrón que ves en los pisos?_(tu respuesta)_

---## Conclusiones generalesRedacta aquí la conclusión del análisis. Dos requisitos:1. Debe ser **condicional** donde los datos lo sean: no "el turístico rinde X%", sino "bajo una ocupación de X, el modelo estima Y".2. Debe distinguir lo que está **medido** de lo que está **asumido**._(tu respuesta)_### Limitaciones_(enumera las que hayas detectado a lo largo del análisis)_